In [7]:
import glob
import zipfile

In [2]:
with zipfile.ZipFile('./GladNet-Dataset.zip', 'r') as f:
    f.extractall('./dataset')

In [8]:
from utils import *
from gladnet import GLADNet

In [9]:
import numpy as np
from PIL import Image
import torch

In [10]:
low = sorted(glob.glob('./dataset/Low/*.png'))
high = sorted(glob.glob('./dataset/Normal/*.png'))

In [11]:
len(low), len(high)

(5000, 5000)

In [12]:
train_low = low[:3000]
train_high = low[:3000]

test_low = low[3000:]
test_high = low[3000:]

In [13]:
model = GLADNet()

In [14]:
img = load_imgs(train_low[0])
img = torch.from_numpy(np.expand_dims(img, 0).transpose((0, 3, 1, 2)))

In [15]:
model(img).shape

torch.Size([1, 3, 400, 600])

In [ ]:
model(train_low)

In [8]:
train_Dataset = CustomDataset(train_low, train_high, train_transform)

In [9]:
test_Dataset = CustomDataset(train_low, train_high, test_transform)

In [10]:
from torch.utils.data import DataLoader

In [11]:
train_loader = DataLoader(train_Dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_Dataset, batch_size=16, shuffle=True)

In [12]:
model = GLADNetCBAM()

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [14]:
loss = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)

In [15]:
from tqdm import tqdm

In [7]:
def check_path(path):
    if not os.path.exists(path):
        os.makedirs(path)


def load_imgs(path):
    img = Image.open(path).convert('RGB')
    img = np.array(img, dtype='float32') / 255.0
    return img


def save_imgs(path, result1, result2=None):
    result1 = np.squeeze(result1)
    result2 = np.squeeze(result2)

    if not result2.any():
        img = result1
    else:
        img = np.concatenate([result1, result2], axis=1)
    img = Image.fromarray(np.clip(img * 255.0, 0, 255.0).astype('uint8'))
    img.save(path, 'png')


class CustomDataset(Dataset):
    def __init__(self, img_paths, label_paths, transform=None):
        self.img_paths = img_paths
        self.label_paths = label_paths
        self.transform = transform

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        img = load_imgs(img_path)
        if self.transform:
            # img = torch.from_numpy(img.transpose((0, 3, 1, 2)))
            # img = torch.from_numpy(img.transpose((-1, 0, 1)))
            img = self.transform(img)
        else:
            img = torch.from_numpy(img.transpose((-1, 0, 1)))
            # img = torch.from_numpy(img)

        if self.label_paths is not None:
            label_path = self.label_paths[index]
            label = load_imgs(label_path)
            if self.transform:
                # label = torch.from_numpy(label.transpose((0, 3, 1, 2)))
                # label = torch.from_numpy(label.transpose((-1, 0, 1)))
                label = self.transform(label)
            else:
                label = torch.from_numpy(label.transpose((-1, 0, 1)))
                # label = torch.from_numpy(label)

            return img, label
        else:
            return img

    def __len__(self):
        return len(self.img_paths)
    

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((96, 96))
])
test_transform = transforms.Compose([
    transforms.ToTensor()
])

In [20]:
load_imgs(train_low[1]).shape

(600, 400, 3)

In [33]:
model

GLADNetCBAM(
  (IDE): IDE(
    (conv1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
      (2): CBAM(
        (ChannelAttention): ChannelAttention(
          (avg_pool): AdaptiveAvgPool2d(output_size=(1, 1))
          (max_pool): AdaptiveAvgPool2d(output_size=(1, 1))
          (fc): Sequential(
            (0): Linear(in_features=64, out_features=4, bias=True)
            (1): ReLU()
            (2): Linear(in_features=4, out_features=64, bias=True)
            (3): ReLU()
          )
        )
        (SpatialAttention): SpatialAttention(
          (conv): Conv2d(64, 1, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1))
        )
      )
    )
    (conv2): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
      (2): CBAM(
        (ChannelAttention): ChannelAttention(
          (avg_pool): AdaptiveAvgPool2d(output_size=(1, 1))
          (max_pool): AdaptiveAvgPool2d(out

In [16]:
model.to(device)
loss.to(device)

best_loss = 0
best_model = None

for epoch in range(51):
    model.train()    
    train_loss = []
    for imgs, labels in tqdm(iter(train_loader)):
        imgs = imgs.float().to(device)
        labels = labels.float().to(device)
        F.interpolate(labels, (96, 96))
        optimizer.zero_grad()
        output = model(imgs)
        criterion = loss(output, labels)
        criterion.backward()
        optimizer.step()

        train_loss.append(criterion.item())
        tr = np.mean(train_loss)
        print(f'Epoch [{epoch}], Train Loss : [{tr:.5f}]]')

        if best_loss >= tr:
            best_loss = tr
            best_model = model

    

  0%|          | 0/188 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
  0%|          | 0/188 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1024x1 and 64x4)

In [21]:
from torchvision import transforms

In [20]:
im = Image.open('./test/Low/a0001-jmac_DSC1459.png').convert('RGB')

In [26]:
transforms.ToTensor()(im).shape

torch.Size([3, 400, 600])

In [12]:
llis = glob.glob('./test/low/*.png')

In [13]:
llis.sort()

In [14]:
llis

['./test/low\\a0001-jmac_DSC1459.png',
 './test/low\\a0002-dgw_005.png',
 './test/low\\a0003-NKIM_MG_8178.png',
 './test/low\\a0004-jmac_MG_1384.png',
 './test/low\\a0005-jn_2007_05_10__564.png',
 './test/low\\a0006-IMG_2787.png',
 './test/low\\a0007-IMG_2480.png',
 './test/low\\a0008-WP_CRW_3959.png',
 './test/low\\a0009-kme_372.png',
 './test/low\\a0010-jmac_MG_4807.png',
 './test/low\\a0011-DSC_0082.png',
 './test/low\\a0012-kme_143.png',
 './test/low\\a0013-MB_20030906_001.png',
 './test/low\\a0014-WP_CRW_6320.png',
 './test/low\\a0015-DSC_0081.png',
 './test/low\\a0016-jmac_MG_0795.png',
 './test/low\\a0017-050710_031618__MG_3496.png',
 './test/low\\a0018-kme_234.png',
 './test/low\\a0019-jmac_MG_0653.png',
 './test/low\\a0020-jmac_MG_6225.png',
 './test/low\\a0021-07-11-28-at-09h22m57s-_MG_7427.png',
 './test/low\\a0022-IMG_2380.png',
 './test/low\\a0023-07-06-02-at-15h06m48-s_MG_1489.png',
 './test/low\\a0024-_DSC8932.png',
 './test/low\\a0025-kme_298.png',
 './test/low\\a0026-k